## Задача


\begin{cases}
\dfrac{dx}{dt} = x \left( 1 - 0.5x - \dfrac{2}{7} \alpha_2^{-2} y \right), \quad x(0) = x_0, \\[10pt]
\dfrac{dy}{dt} = y \left( 2\alpha_2 - 0.5y - 3.5\alpha_2^2 x \right), \quad y(0) = y_0, \\[10pt]
\dfrac{d\alpha_2}{dt} = \varepsilon (2 - 7\alpha_2 x), \quad \alpha_2(0) = \alpha_{20}; \quad t \in [0; T_k].
\end{cases}

#### Рекомендуемые значения начальных данных


$0 \le x_0 \le 3$,

 $0 \le y_0 \le 15$
 
 $\alpha_{20}$ близко к нулю, например,
можно положить $\alpha_{20} = 0.0001$;

 $T_k = 1500$. 
 
$\varepsilon \le 0.01$ 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import seaborn as sns
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d

In [ ]:
def system_func(t, v, epsilon=0.01):
    x, y, a2 = v
    # Защита от деления на ноль для стабильности расчетов
    a2_reg = a2 if abs(a2) > 1e-10 else 1e-10
    
    dxdt = x * (1 - 0.5 * x - (2/7) * (a2_reg**-2) * y)
    dydt = y * (2 * a2 - 0.5 * y - 3.5 * (a2_reg**2) * x)
    da2dt = epsilon * (2 - 7 * a2 * x)
    
    return np.array([dxdt, dydt, da2dt])

In [ ]:
def rk4_step(f, t, y, h):
    k1 = f(t, y)
    k2 = f(t + h/2, y + h/2 * k1)
    k3 = f(t + h/2, y + h/2 * k2)
    k4 = f(t + h, y + h * k3)
    return y + h/6 * (k1 + 2*k2 + 2*k3 + k4)

def euler_explicit(f, y0, t_span, h):
    t = np.arange(t_span[0], t_span[1] + h, h)
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    for i in range(len(t)-1):
        y[i+1] = y[i] + h * f(t[i], y[i])
    return t, y

def implicit_euler(f, y0, t_span, h, tol=1e-7, max_iter=20):
    t = np.arange(t_span[0], t_span[1] + h, h)
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    for i in range(len(t)-1):
        y_guess = y[i] + h * f(t[i], y[i]) # Предиктор
        for _ in range(max_iter):
            y_new = y[i] + h * f(t[i+1], y_guess)
            if np.linalg.norm(y_new - y_guess) < tol: break
            y_guess = y_new
        y[i+1] = y_new
    return t, y

def modified_euler(f, y0, t_span, h):
    t = np.arange(t_span[0], t_span[1] + h, h)
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    for i in range(len(t)-1):
        k1 = f(t[i], y[i])
        k2 = f(t[i+1], y[i] + h * k1)
        y[i+1] = y[i] + (h/2) * (k1 + k2)
    return t, y

def adams_bashforth_4(f, y0, t_span, h):
    t = np.arange(t_span[0], t_span[1] + h, h)
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    # Разгон через RK4
    for i in range(3):
        y[i+1] = rk4_step(f, t[i], y[i], h)
    
    f_vals = [f(t[i], y[i]) for i in range(4)]
    for i in range(3, len(t)-1):
        y[i+1] = y[i] + h/24 * (55*f_vals[3] - 59*f_vals[2] + 37*f_vals[1] - 9*f_vals[0])
        f_vals.pop(0)
        f_vals.append(f(t[i+1], y[i+1]))
    return t, y

def gear_4(f, y0, t_span, h, tol=1e-7):
    t = np.arange(t_span[0], t_span[1] + h, h)
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    for i in range(3): y[i+1] = rk4_step(f, t[i], y[i], h)
    
    for i in range(3, len(t)-1):
        # Начальное приближение (предиктор)
        y_guess = y[i] + h * f(t[i], y[i])
        for _ in range(20):
            # Формула Гира для 4-го порядка (BDF4)
            y_new = (48/25)*y[i] - (36/25)*y[i-1] + (16/25)*y[i-2] - (3/25)*y[i-3] + (12/25)*h*f(t[i+1], y_guess)
            if np.linalg.norm(y_new - y_guess) < tol: break
            y_guess = y_new
        y[i+1] = y_new
    return t, y

In [ ]:
def compute_convergence_orders(methods, system_func, y0, t_span, h_start, n_refinements, ethalon_sol=None):
    """
    methods: словарь { "Имя метода": функция_метода }
    n_refinements: сколько раз уменьшать шаг в 2 раза
    ethalon_sol: если None, считается по правилу Рунге. Если передана функция, то по ней.
    """
    data = []
    prev_results = {name: None for name in methods}
    
    h = h_start
    for i in range(n_refinements):
        row = {'h': h}
        t_eval = np.arange(t_span[0], t_span[1] + h, h)
        
        for name, method_func in methods.items():
            # Вычисляем решение текущим методом
            t_curr, y_curr = method_func(system_func, y0, t_span, h)
            
            # 1. Расчет абсолютной ошибки (Eps)
            if ethalon_sol is not None:
                # Сравнение с эталоном (интерполируем эталон на сетку текущего решения)
                y_true = ethalon_sol(t_curr)
                error = np.max(np.abs(y_curr - y_true))
            else:
                # По правилу Рунге (сравнение с предыдущим шагом)
                if prev_results[name] is not None:
                    # Берем каждое второе значение текущего решения для сравнения с предыдущим
                    error = np.max(np.abs(y_curr[::2] - prev_results[name]))
                else:
                    error = np.nan
            
            row[name] = error
            prev_results[name] = y_curr
            
            # 2. Расчет порядка точности (p)
            p_key = f"p_{name}"
            if i > 0 and not np.isnan(data[i-1][name]) and error > 0:
                # p = log2(ошибка_пред / ошибка_тек)
                order = np.log2(data[i-1][name] / error)
                row[p_key] = round(order, 2)
            else:
                row[p_key] = np.nan
                
        data.append(row)
        h /= 2
        
    return pd.DataFrame(data)

In [ ]:
# Твои параметры
y0_system = [1.5, 7.5, 0.0001]
t_range = [0, 2.0] # Для тестов лучше брать небольшой интервал
eps_param = 0.01

# Решаем систему очень точно
pure_system = lambda t, v: system_func(t, v, eps_param)
sol = solve_ivp(pure_system, t_range, y0_system, method='BDF', atol=1e-12, rtol=1e-12)
ethalon_func = interp1d(sol.t, sol.y, axis=1, kind='cubic', fill_value="extrapolate")

# 2. Собираем методы в словарь
my_methods = {
    "Euler": euler_explicit,
    "ModEuler": modified_euler,
    "Adams4": adams_bashforth_4,
    "Gear4": gear_4
}

# 3. Считаем таблицу
df_results = compute_convergence_orders(
    my_methods, 
    pure_system, 
    y0_system, 
    t_range, 
    h_start=0.01, 
    n_refinements=5, 
    ethalon_sol=lambda t: ethalon_func(t).T
)

print(df_results)

In [ ]:
def analyze_stability(h, v, eps=0.01):
    x, y, a2 = v
    # Матрица Якобиана (J)
    J = np.zeros((3, 3))
    J[0,0] = 1 - x - (2/7)*(a2**-2)*y
    J[0,1] = -(2/7)*(a2**-2)*x
    J[0,2] = (4/7)*(a2**-3)*x*y
    
    J[1,0] = -3.5*(a2**2)*y
    J[1,1] = 2*a2 - y - 3.5*(a2**2)*x
    J[1,2] = 2*y - 7*a2*x*y
    
    J[2,0] = -7*eps*a2
    J[2,1] = 0
    J[2,2] = -7*eps*x
    
    lambdas = np.linalg.eigvals(J)
    
    # 3. Оператор перехода для Эйлера: R = |1 + h*lambda|
    for lam in lambdas:
        z = h * lam
        R_euler = np.abs(1 + z)
        print(f"z = {z:.2f}, |R_euler| = {R_euler:.2f} -> {'OK' if R_euler <= 1 else 'ВЗРЫВ'}")